# Notebook 05 — Presentation Assets (Optional)

**CS19 Capstone Project**

## Scope
This notebook generates visual summary assets from the outputs of Notebooks 01–04.
All assets are saved to `outputs/presentation_assets/`.

Assets produced:
1. `summary_validation_table.png` — MAE comparison table (raw survey / Model C / Proposal 2)
2. `audit_log_screenshot.png` — Proposal 2 time audit log preview
3. `integrated_output_screenshot.png` — Proposal 2 integrated L3 output preview
4. `model_c_integration_pipeline.png` — Pipeline diagram showing where Proposal 2 fits in Model C
5. `implementation_pipeline_diagram.png` — Full implementation pipeline
6. `previous_vs_current_progress.png` — Before/after project progress summary
7. `tester_quality_band_chart.png` — Tester quality band distribution
8. `acc_usa_score_comparison.png` — ACC vs USA score comparison by cohort

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

BASE_DIR   = Path(".").resolve()
OUTPUT_DIR = BASE_DIR / "outputs"
ASSET_DIR  = OUTPUT_DIR / "presentation_assets"
ASSET_DIR.mkdir(parents=True, exist_ok=True)

# Load outputs
summary_csv  = pd.read_csv(OUTPUT_DIR / "proposal2_integration_summary.csv")
audit_csv    = pd.read_csv(OUTPUT_DIR / "proposal2_time_audit_log.csv")
integrated   = pd.read_csv(OUTPUT_DIR / "proposal2_integrated_l3_time_score.csv")
man_summary  = pd.read_csv(OUTPUT_DIR / "manual_review_outputs" / "manual_review_validation_summary.csv")
tq_scores    = pd.read_csv(OUTPUT_DIR / "tester_quality_outputs" / "tester_quality_scores.csv")
p1_rows      = pd.read_csv(OUTPUT_DIR / "proposal1_rows.csv")

print("All output files loaded.")
print(f"Integrated: {integrated.shape}")
print(f"Audit log:  {audit_csv.shape}")
print(f"TQ scores:  {tq_scores.shape}")

All output files loaded.
Integrated: (136, 31)
Audit log:  (14, 16)
TQ scores:  (21, 17)


## 1. Validation summary table

In [2]:
valid_rows = man_summary[man_summary['metric'] != 'tie_cases_multiple_scores_closest'].copy()
valid_rows['metric_label'] = valid_rows['metric'].map({
    'raw_survey_score':    'Raw Survey Score',
    'model_c_before_time': 'Model C Before Time',
    'proposal2_after_time':'Proposal 2 After Time',
})

fig, ax = plt.subplots(figsize=(9, 2.8))
ax.axis('off')
col_labels = ['Metric', 'MAE', 'Median AE', 'N Cases', 'Closest Count (tie-aware)', 'Closest %']
table_data = []
for _, r in valid_rows.iterrows():
    table_data.append([
        r['metric_label'],
        f"{r['mean_absolute_error']:.2f}",
        f"{r['median_absolute_error']:.2f}",
        str(int(r['n_cases'])),
        str(int(r['closest_count_including_ties'])),
        f"{r['closest_percentage_including_ties']:.1f}%",
    ])
t = ax.table(cellText=table_data, colLabels=col_labels, loc='center', cellLoc='center')
t.auto_set_font_size(False)
t.set_fontsize(9)
t.scale(1, 1.6)
for (row, col), cell in t.get_celld().items():
    if row == 0:
        cell.set_facecolor('#2c5f8a')
        cell.set_text_props(color='white', fontweight='bold')
    elif row % 2 == 0:
        cell.set_facecolor('#eef3f8')
    cell.set_edgecolor('#cccccc')
ax.set_title('Manual Review Validation — MAE Comparison (18 cases, 14 valid)', fontsize=11, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig(ASSET_DIR / 'summary_validation_table.png', dpi=150, bbox_inches='tight')
plt.close()
print("summary_validation_table.png saved.")

summary_validation_table.png saved.


## 2. Audit log preview

In [3]:
audit_preview = audit_csv[['email','model_c_task','l3_score_before_time',
                             'proposal2_time_cap','l3_score_after_proposal2_time',
                             'normalized_median_ratio','time_signal']].head(8).copy()
audit_preview.columns = ['Email','Task','L3 Before','Cap','L3 After','Norm Ratio','Signal']
audit_preview['Email'] = audit_preview['Email'].str[:18]
audit_preview['Task']  = audit_preview['Task'].str[:22]
for c in ['L3 Before','L3 After']:
    audit_preview[c] = audit_preview[c].apply(lambda x: f'{float(x):.1f}' if pd.notna(x) else '')
audit_preview['Cap']        = audit_preview['Cap'].apply(lambda x: f'{float(x):.0f}' if pd.notna(x) else '')
audit_preview['Norm Ratio'] = audit_preview['Norm Ratio'].apply(lambda x: f'{float(x):.2f}' if pd.notna(x) else '')

fig, ax = plt.subplots(figsize=(12, 2.5))
ax.axis('off')
t = ax.table(cellText=audit_preview.values, colLabels=audit_preview.columns, loc='center', cellLoc='center')
t.auto_set_font_size(False); t.set_fontsize(8); t.scale(1, 1.5)
for (row, col), cell in t.get_celld().items():
    if row == 0:
        cell.set_facecolor('#2c5f8a'); cell.set_text_props(color='white', fontweight='bold')
    elif row % 2 == 0:
        cell.set_facecolor('#f5f5f5')
    cell.set_edgecolor('#cccccc')
ax.set_title('Proposal 2 Time Audit Log (first 8 cap-triggered rows)', fontsize=10, fontweight='bold', pad=10)
plt.tight_layout()
plt.savefig(ASSET_DIR / 'audit_log_screenshot.png', dpi=150, bbox_inches='tight')
plt.close()
print("audit_log_screenshot.png saved.")

audit_log_screenshot.png saved.


## 3. Integrated output preview

In [4]:
int_preview = integrated[['email','model_c_task','l3_score_before_time',
                            'l3_score_after_proposal2_time','proposal2_changed_score',
                            'time_signal','acc_score_final']].head(8).copy()
int_preview.columns = ['Email','Task','L3 Before','L3 After','Changed','Time Signal','ACC Score']
int_preview['Email'] = int_preview['Email'].str[:18]
int_preview['Task']  = int_preview['Task'].str[:22]
for c in ['L3 Before','L3 After','ACC Score']:
    int_preview[c] = int_preview[c].apply(lambda x: f'{float(x):.1f}' if pd.notna(x) else '')

fig, ax = plt.subplots(figsize=(13, 2.5))
ax.axis('off')
t = ax.table(cellText=int_preview.values, colLabels=int_preview.columns, loc='center', cellLoc='center')
t.auto_set_font_size(False); t.set_fontsize(8); t.scale(1, 1.5)
for (row, col), cell in t.get_celld().items():
    if row == 0:
        cell.set_facecolor('#2c5f8a'); cell.set_text_props(color='white', fontweight='bold')
    elif row % 2 == 0:
        cell.set_facecolor('#f5f5f5')
    cell.set_edgecolor('#cccccc')
ax.set_title('Proposal 2 Integrated L3 Output (first 8 rows)', fontsize=10, fontweight='bold', pad=10)
plt.tight_layout()
plt.savefig(ASSET_DIR / 'integrated_output_screenshot.png', dpi=150, bbox_inches='tight')
plt.close()
print("integrated_output_screenshot.png saved.")

integrated_output_screenshot.png saved.


## 4. Model C integration pipeline diagram

In [5]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.set_xlim(0, 12); ax.set_ylim(0, 5); ax.axis('off')
ax.set_facecolor('#f8f9fa')

boxes = [
    (0.3, 3.2, 1.8, 1.2, '#4a90d9', 'Raw Survey\nScore (1-5)', 'white'),
    (2.5, 3.2, 1.8, 1.2, '#4a90d9', 'Individual\nInsights (LLM)', 'white'),
    (4.7, 3.2, 1.8, 1.2, '#2c5f8a', 'Model C / Proposal 1\nL1 L2 L3 Scoring', 'white'),
    (7.0, 3.2, 1.8, 1.2, '#27ae60', 'Proposal 2\nTime Signal', 'white'),
    (9.2, 3.2, 1.8, 1.2, '#e67e22', 'L3.8 Time\nModeration', 'white'),
    (4.7, 0.8, 1.8, 1.2, '#8e44ad', 'Accessibility\nScore (L2)', 'white'),
    (7.0, 0.8, 1.8, 1.2, '#c0392b', 'Usability Score\n(L3 + L3.8)', 'white'),
    (9.2, 0.8, 1.8, 1.2, '#7f8c8d', 'Tester Quality\n(Independent)', 'white'),
]
for x, y, w, h, color, label, tc in boxes:
    rect = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.05',
                           facecolor=color, edgecolor='white', linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x+w/2, y+h/2, label, ha='center', va='center',
            fontsize=8, color=tc, fontweight='bold', wrap=True)

arrows = [
    (2.1, 3.8, 2.5, 3.8), (4.3, 3.8, 4.7, 3.8), (6.5, 3.8, 7.0, 3.8),
    (8.8, 3.8, 9.2, 3.8), (5.6, 3.2, 5.6, 2.0), (8.1, 3.2, 7.9, 2.0),
    (9.2+0.9, 3.2, 9.2+0.9, 2.0),
]
for x1,y1,x2,y2 in arrows:
    ax.annotate('', xy=(x2,y2), xytext=(x1,y1),
                arrowprops=dict(arrowstyle='->', color='#555', lw=1.5))

ax.text(6.0, 4.7, 'CS19 Model C Integration Pipeline', ha='center', va='center',
        fontsize=12, fontweight='bold', color='#2c3e50')
ax.text(5.6, 0.3, 'Note: Proposal 2 only changes L3 usability. ACC scores are unchanged.',
        ha='center', va='center', fontsize=8, color='#555', style='italic')

plt.tight_layout()
plt.savefig(ASSET_DIR / 'model_c_integration_pipeline.png', dpi=150, bbox_inches='tight')
plt.close()
print("model_c_integration_pipeline.png saved.")

model_c_integration_pipeline.png saved.


## 5. Implementation pipeline diagram

In [6]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.set_xlim(0, 12); ax.set_ylim(0, 4); ax.axis('off')
ax.set_facecolor('#f8f9fa')

steps = [
    (0.2, 1.5, 1.8, 1.0, '#3498db', 'NB 01\nModel C / P1'),
    (2.4, 1.5, 1.8, 1.0, '#27ae60', 'NB 02\nProposal 2'),
    (4.6, 1.5, 1.8, 1.0, '#e67e22', 'NB 03\nManual Review'),
    (6.8, 1.5, 1.8, 1.0, '#8e44ad', 'NB 04\nTester Quality'),
    (9.0, 1.5, 1.8, 1.0, '#c0392b', 'NB 05\nPresentation'),
]
for x, y, w, h, color, label in steps:
    rect = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.05',
                           facecolor=color, edgecolor='white', linewidth=2)
    ax.add_patch(rect)
    ax.text(x+w/2, y+h/2, label, ha='center', va='center',
            fontsize=9, color='white', fontweight='bold')

for i in range(len(steps)-1):
    x1 = steps[i][0] + steps[i][2]
    x2 = steps[i+1][0]
    y  = steps[i][1] + steps[i][3]/2
    ax.annotate('', xy=(x2, y), xytext=(x1, y),
                arrowprops=dict(arrowstyle='->', color='#555', lw=2))

outputs = [
    (0.2, 0.2, 'proposal1_rows.csv\nmodel_c3_proposal1_results.xlsx'),
    (2.4, 0.2, 'proposal2_integrated_l3_time_score.csv\nproposal2_time_audit_log.csv'),
    (4.6, 0.2, 'manual_review_cases.csv\nmanual_review_validation_summary.csv'),
    (6.8, 0.2, 'tester_quality_scores.csv\ntester_quality_features.csv'),
    (9.0, 0.2, 'PNG assets\n(6 charts)'),
]
for x, y, label in outputs:
    ax.text(x+0.9, y+0.4, label, ha='center', va='center',
            fontsize=6.5, color='#333', style='italic')
    ax.annotate('', xy=(x+0.9, y+0.8), xytext=(x+0.9, 1.5),
                arrowprops=dict(arrowstyle='->', color='#aaa', lw=1))

ax.text(6.0, 3.6, 'CS19 Capstone — Notebook Implementation Pipeline', ha='center', va='center',
        fontsize=11, fontweight='bold', color='#2c3e50')

plt.tight_layout()
plt.savefig(ASSET_DIR / 'implementation_pipeline_diagram.png', dpi=150, bbox_inches='tight')
plt.close()
print("implementation_pipeline_diagram.png saved.")

implementation_pipeline_diagram.png saved.


## 6. Before/after progress summary

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
fig.suptitle('CS19 Capstone — Project Progress Summary', fontsize=12, fontweight='bold')

# Left: score comparison
tasks = p1_rows.groupby('task')[['acc_score_final','usa_score_final']].mean().reset_index()
tasks['task_short'] = tasks['task'].str[:25]
x = np.arange(len(tasks))
w = 0.35
axes[0].bar(x-w/2, tasks['acc_score_final'], w, label='Accessibility (ACC)', color='#2c5f8a', alpha=0.85)
axes[0].bar(x+w/2, tasks['usa_score_final'],  w, label='Usability (USA)',     color='#27ae60', alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(tasks['task_short'], rotation=20, ha='right', fontsize=7)
axes[0].set_ylabel('Mean Score (0-100)')
axes[0].set_title('Model C / Proposal 1 — Task-Level Scores')
axes[0].legend(fontsize=8)
axes[0].set_ylim(0, 100)
axes[0].axhline(50, color='red', linestyle='--', linewidth=0.8, alpha=0.5, label='50 threshold')

# Right: Proposal 2 impact
changed = integrated[integrated['proposal2_changed_score'].astype(str).str.lower()=='true'].copy()
if len(changed) > 0:
    axes[1].scatter(changed['l3_score_before_time'], changed['l3_score_after_proposal2_time'],
                    c='#e67e22', s=60, alpha=0.8, label='Changed scores')
unchanged = integrated[integrated['proposal2_changed_score'].astype(str).str.lower()!='true'].copy()
axes[1].scatter(unchanged['l3_score_before_time'], unchanged['l3_score_after_proposal2_time'],
                c='#3498db', s=20, alpha=0.4, label='Unchanged scores')
axes[1].plot([0,100],[0,100], 'k--', linewidth=0.8, alpha=0.5)
axes[1].set_xlabel('L3 Before Time (Proposal 1)')
axes[1].set_ylabel('L3 After Time (Proposal 2)')
axes[1].set_title('Proposal 2 — L3 Score Impact')
axes[1].legend(fontsize=8)
axes[1].set_xlim(0,100); axes[1].set_ylim(0,100)

plt.tight_layout()
plt.savefig(ASSET_DIR / 'previous_vs_current_progress.png', dpi=150, bbox_inches='tight')
plt.close()
print("previous_vs_current_progress.png saved.")

previous_vs_current_progress.png saved.


## 7. Tester quality band chart

In [8]:
band_order = ['high_quality','good_quality','medium_quality','low_quality_review_needed','insufficient_data']
band_colors = {'high_quality':'#27ae60','good_quality':'#3498db',
               'medium_quality':'#f39c12','low_quality_review_needed':'#e74c3c','insufficient_data':'#95a5a6'}
band_counts = tq_scores['tester_quality_band'].value_counts().reindex(band_order, fill_value=0)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(band_counts.index, band_counts.values,
              color=[band_colors[b] for b in band_counts.index], edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, band_counts.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1, str(val),
            ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_xticklabels([b.replace('_',' ').title() for b in band_counts.index], rotation=15, ha='right', fontsize=9)
ax.set_ylabel('Number of Testers')
ax.set_title('Tester Quality Rating — Band Distribution (n=21)', fontsize=11, fontweight='bold')
ax.set_ylim(0, max(band_counts.values)+2)
ax.text(0.98, 0.97, 'Independent third score — does not change ACC/USA scores',
        transform=ax.transAxes, ha='right', va='top', fontsize=7, color='#888', style='italic')
plt.tight_layout()
plt.savefig(ASSET_DIR / 'tester_quality_band_chart.png', dpi=150, bbox_inches='tight')
plt.close()
print("tester_quality_band_chart.png saved.")

/tmp/ipykernel_3803/2808223831.py:12: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels([b.replace('_',' ').title() for b in band_counts.index], rotation=15, ha='right', fontsize=9)


tester_quality_band_chart.png saved.


## 8. ACC vs USA score comparison by cohort

In [9]:
cohort_summary = (
    p1_rows.groupby('primary_cohort', dropna=False)
    [['acc_score_final','usa_score_final']]
    .mean().reset_index()
    .sort_values('acc_score_final')
)
cohort_summary['cohort_label'] = cohort_summary['primary_cohort'].fillna('Unknown')

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(cohort_summary))
w = 0.35
ax.bar(x-w/2, cohort_summary['acc_score_final'], w, label='Accessibility (ACC)', color='#2c5f8a', alpha=0.85)
ax.bar(x+w/2, cohort_summary['usa_score_final'],  w, label='Usability (USA)',     color='#27ae60', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(cohort_summary['cohort_label'], rotation=15, ha='right', fontsize=9)
ax.set_ylabel('Mean Score (0-100)')
ax.set_title('Model C / Proposal 1 — ACC vs USA by Cohort', fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
ax.set_ylim(0, 100)
ax.axhline(50, color='red', linestyle='--', linewidth=0.8, alpha=0.4)
plt.tight_layout()
plt.savefig(ASSET_DIR / 'acc_usa_score_comparison.png', dpi=150, bbox_inches='tight')
plt.close()
print("acc_usa_score_comparison.png saved.")

acc_usa_score_comparison.png saved.


In [10]:
print("\nAll presentation assets saved to:", ASSET_DIR)
for f in sorted(ASSET_DIR.glob('*.png')):
    print(f"  {f.name}: {f.stat().st_size:,} bytes")


All presentation assets saved to: /home/ubuntu/cs19_capstone/outputs/presentation_assets
  acc_usa_score_comparison.png: 39,247 bytes
  audit_log_screenshot.png: 96,981 bytes
  implementation_pipeline_diagram.png: 63,411 bytes
  integrated_output_screenshot.png: 88,208 bytes
  model_c_integration_pipeline.png: 66,900 bytes
  previous_vs_current_progress.png: 102,041 bytes
  summary_validation_table.png: 48,494 bytes
  tester_quality_band_chart.png: 62,843 bytes
